In [1]:
!pip install langchain langchain-community chromadb openai
!pip install -q langchain_huggingface sentence_transformers --no-deps
!pip install -U langchain-huggingface
!pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 88.9 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 87.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 92.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 89.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [chromadb]/24 [chromadb]s]]
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.7
    Uninstalling langchain-core-1.2.7:
      Successfully uninstalled langchain-core-1.2.7


In [2]:
import glob

# PARTIE 1 (retrieval)
from langchain_community.document_loaders import PyPDFLoader

# Initialisation du loader de document pour charger un fichier PDF
documents = []

# Chaque page devient un objet Document contenant :
# 	• 	page_content → le texte
# 	•	metadata → source, page, auteur, etc.

for file in glob.glob("DIC/*.pdf"):
    try:
        loader = PyPDFLoader(file)  # Retourne une liste de document (un pour chaque page)
        documents += loader.load()
    except Exception as e:
        print(f"Erreur survenue pour le fichier '{file}' : {e}")

In [3]:
# Première page du premier document PDF
documents[0]

Document(metadata={'producer': 'Actuate PDF Writer (Low Resolution) 2.1', 'creator': 'Actuate e.Reports', 'creationdate': '2022-07-29T08:11:45+01:00', 'title': '', 'subject': '', 'author': 'IDS GmbH - Analysis and Reporting Services', 'keywords': 'FR0010032326 (22.08.2022)', 'source': 'DIC/Allianz.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='\x00I\x00n\x00f\x00o\x00r\x00m\x00a\x00t\x00i\x00o\x00n\x00s\x00 \x00c\x00l\x00Ø\x00s\x00 \x00p\x00o\x00u\x00r\x00 \x00l \x19\x00i\x00n\x00v\x00e\x00s\x00t\x00i\x00s\x00s\x00e\x00u\x00r\n\x00C\x00e\x00 \x00d\x00o\x00c\x00u\x00m\x00e\x00n\x00t\x00 \x00f\x00o\x00u\x00r\x00n\x00i\x00t\x00 \x00d\x00e\x00s\x00 \x00i\x00n\x00f\x00o\x00r\x00m\x00a\x00t\x00i\x00o\x00n\x00s\x00 \x00e\x00s\x00s\x00e\x00n\x00t\x00i\x00e\x00l\x00l\x00e\x00s\x00 \x00a\x00u\x00x\x00 \x00i\x00n\x00v\x00e\x00s\x00t\x00i\x00s\x00s\x00e\x00u\x00r\x00s\x00 \x00d\x00e\x00 \x00c\x00e\x00t\x00 \x00O\x00P\x00C\x00V\x00M\x00.\x00 \x00I\x00l\x00 \x00n\x00e\x00 \x00s

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Initialisation du séparateur de texte avec des paramètres spécifiques pour diviser le texte
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,  # Taille maximale des morceaux de texte
    chunk_overlap=60,  # Chevauchement entre les morceaux pour garder le contexte
    length_function=len,  # Fonction pour calculer la longueur des morceaux
    separators=["\n\n", "\n"]  # Séparateurs utilisés pour diviser le texte en morceaux
)

# Division du document en morceaux (chunks)
chunks = text_splitter.split_documents(documents=documents)

# Affichage du nombre de morceaux créés à partir du document PDF
print(f"{len(chunks)} chunks ont été créés par le splitter à partir des documents PDF du dossier DIC.")

1721 chunks ont été créés par le splitter à partir des documents PDF du dossier DIC.


In [5]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [6]:
# Choisir un modèle d’embedding
#On normalise les embeddings afin d’améliorer la qualité et la stabilité du calcul de similarité (notamment la similarité cosinus) lors de la recherche vectorielle.
#modèle open-source multilingue car il comprend bien le français et offre de bonnes performances en recherche sémantique, tout en étant exécutable localement sans API externe.
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2", encode_kwargs={"normalize_embeddings" : True})

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
#Créer la base Chroma (locale)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",  # stockage local
    collection_name="dic_collection_v3"  
)

In [8]:
#sauvegarde
vectorstore.persist()

/tmp/ipykernel_131/2251463224.py:2: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [9]:
#vérification  Test de recherche sémantique
query = "Que montre le scénario de tensions ?"
results = vectorstore.similarity_search(query)

for r in results:
    print(r.page_content)

 F r a i s   d ’ e n t r Ø e  2 , 0 0   %
 F r a i s   d e   s o r t i e  0 , 0 0   %
 L e   p o u r c e n t a g e   i n d i q u Ø   e s t   l e   m a x i m u m   p o u v a n t   Œ t r e   p r Ø l e v Ø   s u r   v o t r e
 c a p i t a l   a v a n t   q u e   c e l u i - c i   n e   s o i t   i n v e s t i .   D a n s   c e r t a i n s   c a s   l  i n v e s t i s s e u r
 p e u t   p a y e r   m o i n s .   L ’ i n v e s t i s s e u r   p e u t   o b t e n i r   d e   s o n   c o n s e i l l e r   o u   d e   s o n
 F r a i s   d ’ e n t r Ø e  2 , 0 0   %
 F r a i s   d e   s o r t i e  0 , 0 0   %
 L e   p o u r c e n t a g e   i n d i q u Ø   e s t   l e   m a x i m u m   p o u v a n t   Œ t r e   p r Ø l e v Ø   s u r   v o t r e
 c a p i t a l   a v a n t   q u e   c e l u i - c i   n e   s o i t   i n v e s t i .   D a n s   c e r t a i n s   c a s   l  i n v e s t i s s e u r
 p e u t   p a y e r   m o i n s .   L ’ i n v e s t i s s e u r   p e u t   o b t e n i r   d e   s 

In [10]:
## PARTIE 2 
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

# Modèle 7B choisi afin d’assurer un bon équilibre entre qualité de génération,
# stabilité GPU (24GB VRAM) et performance en temps de réponse,
# largement suffisant pour un RAG d’analyse factuelle de documents DIC.
# dans une architecture RAG,
# la qualité dépend principalement du retrieval ; un modèle plus volumineux (14B)
# n’apporte pas de gain significatif pour l’analyse factuelle des DIC
# mais augmente fortement la consommation mémoire et le temps d’inférence.

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

print(model.config._name_or_path)

# 256 tokens maximum afin d’optimiser le temps d’inférence
# et prévenir les dépassements mémoire (OOM) observés avec des limites plus élevées.

generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    return_full_text=False
)

# Création d'une instance de HuggingFacePipeline à partir de l'identifiant du modèle spécifié (LangChain wrapper)
llm = HuggingFacePipeline(pipeline=generation_pipeline)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


mistralai/Mistral-7B-Instruct-v0.2


In [24]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="history",
    return_messages=False
)

prompt_template = """Tu es un assistant spécialisé en analyse de documents d’informations clés (DIC) pour un département ALM d’assurance vie.Réponds uniquement à partir du contexte fourni.
Tu ne dois jamais utiliser tes connaissances internes.
Réponds uniquement à partir du contexte fourni.

Tu dois obligatoirement citer la source exacte du document utilisé pour formuler la réponse.

Si aucune source n’est trouvée dans le contexte fourni,
indique explicitement :
"Information non disponible dans les DIC fournis."

Historique de la conversation :
{history}

Contexte :
{context}

Question :
{question}

Réponse (réponds uniquement à la question posée, sans ajouter de nouvelle question) :
"""

def rag_pipeline(query, use_eval=False):
    # Choix du vectorstore
    current_vectorstore = vectorstore_eval if use_eval else vectorstore
    
    # Récupérer l'historique
    history = memory.load_memory_variables({})["history"]
    
    # on recherche les documents
    # Limitation à 2 documents récupérés pour contrôler la longueur du prompt,
    # diminuer la charge GPU et améliorer la stabilité du pipeline RAG.
    retrieved_docs = current_vectorstore .similarity_search(query, k=2)

    context="\n\n".join(
            f"Source: {doc.metadata.get('source', 'Inconnue')} - Page {doc.metadata.get('page', '?')}\n{doc.page_content}"
            for doc in retrieved_docs
    )
    
    # Ensuite, on injecte l'historique et les documents (avec leurs metadata) dans le prompt
    prompt = prompt_template.format(
        history=history,
        question=query,
        context=context
    )
    # Appel via llm (PAS pipeline direct)
    response = llm.invoke(prompt, stop=["Question:"])
    
    # Sauvegarde dans la mémoire
    memory.save_context({"input": query}, {"output": response})
    
    return response

In [25]:
query = """
Quel est l'objectif de gestion du FCP décrit dans le document ? ?
"""

# Effectuer une requête
response = rag_pipeline(query)
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



L'objectif de gestion du FCP décrit dans le document est de mettre en œuvre une gestion discrétionnaire pour surperformer, via une exposition aux actions du secteur immobilier de l'Union européenne, en conciliant performance financière et extra-financière, l'indicateur de référence FTSE EPRA / NAREIT Euro Zone Capped dividendes nets réinvestis après déduction des frais de gestion sur la durée de placement recommandée (supérieure à 5 ans).

Source : DIC/SODIFY.pdf - Page 0.


In [26]:
def chat():
    print("Assistant DIC prêt. Tape 'exit' pour quitter.\n")
    
    while True:
        user_input = input("Vous : ")
        
        if user_input.lower() in ["exit", "quit"]:
            print("Fin de la conversation.")
            break
        
        response = rag_pipeline(user_input)
        print("\nAssistant :", response, "\n")

In [28]:
memory.clear()
chat()

Assistant DIC prêt. Tape 'exit' pour quitter.



Vous :  exit


Fin de la conversation.


In [29]:
## PARTIE 3 - Evaluation

#   1.	recréer la base de test à partir de corpus.json (chunks),
# === Dépendances (installer si nécessaire) ===
# pip install chromadb sentence-transformers langchain tqdm

import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# === 1️⃣ Charger le corpus ===
DATASET_FOLDER_PATH = Path("dataset_eval")
with open(base / "corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Nombre de chunks : {len(corpus)}")

# === 2️⃣ Transformer en Documents ===
documents = [
    Document(page_content=text, metadata={"uuid": uid})
    for uid, text in corpus.items()
]

print(f"{len(documents)} Documents prêts.")

# === 3️⃣ Embeddings ===
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# === 4️⃣ Vectorstore ===
persist_directory = "./chroma_eval_db"

vectorstore_eval = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="evaluation_collection"
)

vectorstore_eval.persist()

print("Base d’évaluation construite avec succès.")

Nombre de chunks : 250
250 Documents prêts.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Base d’évaluation construite avec succès.


In [ ]:
#	2.	évaluer le retrieval (Precision@k, Recall@k, MRR),
#	3.	évaluer la génération (BERTScore F1 vs answers.json),

import json
from pathlib import Path
import os
import csv
from tqdm import tqdm
from bert_score import score as bert_score_fn
import torch

# Réglages
K = 2   # k utilisé pour similarity_search
BERT_THRESHOLD = 0.60  # seuil accepté pour le F1 BERTScore (60%)
LANG = "fr"  

# Charger fichiers
with open(DATASET_FOLDER_PATH / "queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)  # dict uuid -> query_text

with open(DATASET_FOLDER_PATH / "answers.json", "r", encoding="utf-8") as f:
    answers = json.load(f)  # dict uuid -> expected_answer_text

with open(DATASET_FOLDER_PATH / "relevant_docs.json", "r", encoding="utf-8") as f:
    relevant_docs = json.load(f)  # dict uuid -> [list of relevant doc uuids]

# Prépare stockage des résultats
results = []

# Boucle d'évaluation, on boule sur les uuid qui sont présents dans queries, answers et relevant_docs
common_uuids = (
    set(queries.keys())
    .intersection(set(answers.keys()))
    .intersection(set(relevant_docs.keys()))
)
print(f"Evaluation de {len(common_uuids)} requêtes (K={K}) — device torch: {torch.cuda.is_available() and 'cuda' or 'cpu'}")

# tqdm sert à visualiser la progression d’une boucle.
for uid in tqdm(common_uuids, desc="Evaluation RAG"):
    query = queries[uid].strip()
    ref_answer = answers[uid].strip()

    # 1) générer la réponse via ta pipeline d'éval
    try:
        raw_gen_answer = rag_pipeline(query, use_eval=True)
        # Si rag_pipeline renvoie un objet complexe, extraire string :
        if isinstance(raw_gen_answer, (list, tuple)):
            gen_answer = " ".join(str(x) for x in raw_gen_answer)
        else:
            gen_answer = str(raw_gen_answer)
    except Exception as e:
        gen_answer = ""
        print(f"[ERROR] génération pour {uid} a levé : {e}")

    # 2) retrieval (même comportement que dans la pipeline)
    retrieved_docs = vectorstore_eval.similarity_search(query, k=K)

    # Extraire identifiants/uuids des docs récupérés — on cherche un champ 'uuid' ou 'source' dans metadata
    # Chaque chunk indexé dans Chroma a son uuid du corpus.json.
    retrieved_uuids = [doc.metadata["uuid"] for doc in retrieved_docs]

    # 3) comparer aux relevant_docs attendus (liste)
    expected_uuids = relevant_docs[uid]
    # Le recall mesure la proportion des documents réellement pertinents
    # (définis dans relevant_docs.json) qui ont été correctement récupérés
    # par le moteur de recherche du pipeline RAG.
    if len(expected_uuids) == 0:
        recall_at_k = None
    else:
        found = sum(1 for e in expected_uuids if str(e) in retrieved_uuids)
        recall_at_k = found / len(expected_uuids)

    results.append({
        "uuid": uid,
        "query": query,
        "gen": gen_answer,
        "ref": ref_answer,
        "retrieved_uuids": retrieved_uuids,
        "expected_uuids": expected_uuids,
        "recall_at_k": recall_at_k,
    })

# 4) Calcul du BERTScore (en batch pour accélérer)
# Le BERTScore évalue la proximité sémantique entre deux textes à l’aide d’un modèle BERT.
print("Calcul du BERTScore (F1) — ceci peut prendre quelques secondes/minutes selon la GPU/CPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"

gen_answers = [r["gen"] for r in results]
ref_answers  = [r["ref"] for r in results]

# Le BERTScore utilisé dans cette évaluation correspond au F1,
# qui combine la précision et le rappel sémantiques entre la réponse générée et la référence.
P, R, F1 = bert_score_fn(cands=gen_answers,
                         refs=ref_answers,
                         lang=LANG,
                         model_type=None,  # laisser bert-score choisir le modèle adapté à la langue
                         device=device,
                         verbose=True)

# bert_score renvoie tensors — convertir en float
F1_list = [float(x) for x in F1]

# Remplir les résultats avec le F1
for i, uid in enumerate(common_uuids):
    results[i]["bert_f1"] = F1_list[i]

# 5) métriques agrégées
import statistics
valid_f1s = [v for v in F1_list if not (v is None or (isinstance(v, float) and (v != v)))]
mean_f1 = statistics.mean(valid_f1s) if valid_f1s else 0.0
# %des query qui ont F1 > 60%
pct_above_threshold = 100.0 * sum(1 for v in valid_f1s if v >= BERT_THRESHOLD) / len(valid_f1s) if valid_f1s else 0.0

# calcul de la moyenne du recall
recalls = [r["recall_at_k"] for r in results if r["recall_at_k"] is not None]
mean_recall_at_k = statistics.mean(recalls) if recalls else None

print("\n--- Résultats agrégés ---")
print(f"Nombre de requêtes: {len(uuids)}")
print(f"Mean BERT-F1 : {mean_f1:.4f}")
print(f"% exemples avec BERT-F1 >= {BERT_THRESHOLD*100:.0f}% : {pct_above_threshold:.2f}%")
print(f"Mean recall@{K} (sur les requêtes ayant des relevant_docs) : {mean_recall_at_k if mean_recall_at_k is not None else 'N/A'}")

# 6) sauver résultats détaillés en CSV
OUT = DATA_DIR / "evaluation_results.csv"
with open(OUT, "w", encoding="utf-8", newline="") as csvfile:
    fieldnames = ["uuid","query","gen","ref","bert_f1","retrieved_uuids","expected_uuids","recall_at_k"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({
            "uuid": r["uuid"],
            "query": r["query"],
            "gen": r["gen"],
            "ref": r["ref"],
            "bert_f1": r.get("bert_f1"),
            "retrieved_uuids": "|".join(r["retrieved_uuids"]),
            "expected_uuids": "|".join([str(x) for x in r["expected_uuids"]]),
            "recall_at_k": r["recall_at_k"]
        })

print(f"Résultats sauvegardés dans : {OUT}")


#	4.	agréger les résultats et vérifier si F1 ≥ 60%,
#	5.	produire un errors.json si nécessaire.

Evaluation de 619 requêtes (K=2) — device torch: cuda


Eval:   0%|          | 0/619 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Eval:   0%|          | 1/619 [00:06<1:09:40,  6.76s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Eval:   0%|          | 2/619 [00:07<31:28,  3.06s/it]  Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentat

In [ ]:
from datetime import datetime
import re

# Nettoyer le nom du modèle pour qu'il soit compatible fichier
safe_model_name = re.sub(r"[^\w\-]", "_", model_id.split("/")[-1])

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

SUMMARY_OUT = DATA_DIR / f"evaluation_summary_{safe_model_name}_k{K}_thr{int(BERT_THRESHOLD*100)}_{timestamp}.csv"
DETAIL_OUT = DATA_DIR / f"evaluation_results_{safe_model_name}_k{K}_thr{int(BERT_THRESHOLD*100)}_{timestamp}.csv"

# 1️⃣ Sauvegarde détaillée
with open(DETAIL_OUT, "w", encoding="utf-8", newline="") as csvfile:
    fieldnames = [
        "uuid","query","gen","ref","bert_f1",
        "retrieved_uuids","expected_uuids","recall_at_k"
    ]
    
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    for r in results:
        writer.writerow({
            "uuid": r["uuid"],
            "query": r["query"],
            "gen": r["gen"],
            "ref": r["ref"],
            "bert_f1": r.get("bert_f1"),
            "retrieved_uuids": "|".join(r["retrieved_uuids"]),
            "expected_uuids": "|".join([str(x) for x in r["expected_uuids"]]),
            "recall_at_k": r["recall_at_k"]
        })

# 2️⃣ Sauvegarde résumé
with open(SUMMARY_OUT, "w", encoding="utf-8", newline="") as csvfile:
    fieldnames = [
        "model_name",
        "k",
        "threshold",
        "nb_queries",
        "mean_bert_f1",
        "pct_above_threshold",
        f"mean_recall_at_{K}"
    ]
    
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    writer.writerow({
        "model_name": model_id,
        "k": K,
        "threshold": BERT_THRESHOLD,
        "nb_queries": len(uuids),
        "mean_bert_f1": round(mean_f1, 4),
        "pct_above_threshold": round(pct_above_threshold, 2),
        f"mean_recall_at_{K}": round(mean_recall_at_k, 4) if mean_recall_at_k is not None else "N/A"
    })

print(f"Détail sauvegardé dans : {DETAIL_OUT}")
print(f"Résumé sauvegardé dans : {SUMMARY_OUT}")